# Robustness - part 3: notice drift before the accuracy does

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomasvicar/AUI-public/blob/master/labs/robustness/notebooks/ex5_monitoring.ipynb)

**Computer-Aided Medical Diagnostics (MPA-PRM)** · lab for the lecture
*Robustness, generalization and reliability*

The Heart Disease model goes into service. Every week the clinic scores 60 new
patients; the diagnoses arrive weeks later. From week 6 a new referral pathway
starts sending **sicker patients**. You watch only what you have every week -
the model's predicted probabilities - and ask two questions: *how much* did
their distribution move (PSI - the population stability index), and *how
sure* are we that it moved at all (the Kolmogorov-Smirnov test)?

**Three holes.** PSI by hand, the weekly dashboard - and a second scenario,
patients of a new hospital, with one more monitor next to PSI and K-S.

# Data and model

The same data, split and XGBoost as in part 2.

In [ ]:
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.optimize import minimize_scalar
from scipy.stats import ks_2samp
from sklearn.model_selection import train_test_split

URL = "https://raw.githubusercontent.com/tomasvicar/AUI-public/master/labs/robustness/data/heart_disease_cleveland.csv"
CSV = Path("heart_disease_cleveland.csv")
if not CSV.is_file():
    urllib.request.urlretrieve(URL, CSV)

SEED = 42
N_BINS = 5


def load_heart(path: Path = CSV) -> tuple[pd.DataFrame, pd.Series]:
    """13 features (missing values filled with the median) and disease = num > 0."""
    table = pd.read_csv(path)
    y = (table.pop("num") > 0).astype(int)
    return table.fillna(table.median(numeric_only=True)), y


def split(X: pd.DataFrame, y: pd.Series, seed: int = SEED):
    """Train / calibration / test = 50 / 25 / 25 %, stratified."""
    X_train, X_rest, y_train, y_rest = train_test_split(
        X, y, test_size=0.5, random_state=seed, stratify=y)
    X_calib, X_test, y_calib, y_test = train_test_split(
        X_rest, y_rest, test_size=0.5, random_state=seed, stratify=y_rest)
    return X_train, X_calib, X_test, y_train, y_calib, y_test


def train_xgboost(X_train: pd.DataFrame, y_train: pd.Series, seed: int = SEED):
    """The shared recipe of the tabular running example."""
    model = xgb.XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1,
                              eval_metric="logloss", random_state=seed)
    return model.fit(X_train, y_train)


def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-z))

In [ ]:
X, y = load_heart()
X_train, X_calib, X_test, y_train, y_calib, y_test = split(X, y)
model = train_xgboost(X_train, y_train)
print(f"train {len(X_train)}, calibration {len(X_calib)}, test {len(X_test)} patients")

p_test = model.predict_proba(X_test)[:, 1]
print(f"test accuracy {((p_test > 0.5) == y_test).mean():.3f}")

# Ten weeks of patients

Each week bootstraps 60 patients from the test part. Weeks 1-5 keep its case
mix (46 % sick); from week 6 the share of sick patients grows by ten points a
week, up to 95 %. The reference is the model's predictions on the whole test
part - the distribution it was validated on.

In [ ]:
N_WEEKS = 10
DRIFT_START_WEEK = 6
BATCH_SIZE = 60
PREVALENCE_STEP = 0.10
MAX_PREVALENCE = 0.95
PSI_BINS = 5
WATCH, ALERT = 0.10, 0.25


def weekly_prevalence(week: int, base: float) -> float:
    """The share of sick patients the clinic sends in a given week."""
    return min(MAX_PREVALENCE, base + PREVALENCE_STEP * max(0, week - DRIFT_START_WEEK + 1))


def weekly_batches(X_pool: pd.DataFrame, y_pool: np.ndarray, seed: int = SEED):
    """Ten weekly batches of 60 patients bootstrapped from the pool. Weeks 1-5
    keep the pool's case mix; from week 6 a new referral pathway sends sicker
    patients - the prevalence grows by 10 points a week (label shift)."""
    rng = np.random.RandomState(seed)
    base = y_pool.mean()
    batches = []
    for week in range(1, N_WEEKS + 1):
        prevalence = weekly_prevalence(week, base)
        weights = np.where(y_pool == 1, prevalence / base, (1 - prevalence) / (1 - base))
        index = rng.choice(len(X_pool), BATCH_SIZE, replace=True, p=weights / weights.sum())
        batches.append((X_pool.iloc[index].reset_index(drop=True), y_pool[index]))
    return batches


def status(value: float) -> str:
    return "alert" if value >= ALERT else "watch" if value >= WATCH else "ok"


y_pool = y_test.to_numpy()
reference = model.predict_proba(X_test)[:, 1]
batches = weekly_batches(X_test, y_pool)
print(f"{len(batches)} weeks of {BATCH_SIZE} patients; reference: "
      f"{len(reference)} predictions on the test part")

### Hole 1 - the population stability index

In [ ]:
# TODO 1: the population stability index of two samples of predicted
#         probabilities, on PSI_BINS equal-width bins of [0, 1]:
#
#   PSI = sum over bins of (q_i - p_i) * ln(q_i / p_i)
#
# p_i, q_i are the SHARES of the reference and the current sample in bin i
# (np.histogram with np.linspace(0, 1, n_bins + 1) as edges). An empty bin
# would give log(0): give every share at least 1e-4 (np.clip).

def psi(reference, current, n_bins=PSI_BINS):
    ...


print(f"PSI of the reference against itself: {psi(reference, reference):.3f}")

### Hole 2 - the dashboard

In [ ]:
# TODO 2: for every week - the prevalence, PSI and its status, the K-S p-value
#         (scipy.stats.ks_2samp(reference, p).pvalue) and the accuracy the
#         clinic would only learn weeks later. One dict per week in `weeks`,
#         with the keys the dashboard below reads:
#         {"week", "prevalence", "psi", "status", "ks_p", "accuracy"}
#         (`weekly_prevalence` and `status` are defined above).

weeks = []
for week, (batch, labels) in enumerate(batches, start=1):
    ...

In [ ]:
colors = {"ok": "teal", "watch": "orange", "alert": "red"}
w = [x["week"] for x in weeks]
fig, (top, bottom) = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)
top.bar(w, [x["psi"] for x in weeks], color=[colors[x["status"]] for x in weeks])
top.axhline(WATCH, color="k", ls="--", lw=1); top.axhline(ALERT, color="k", ls="--", lw=1)
top.set_ylabel("PSI")
bottom.semilogy(w, [x["ks_p"] for x in weeks], "o-k")
bottom.axhline(0.05, color="red", ls="--", lw=1)
bottom.set_ylabel("K-S p-value"); bottom.set_xticks(w)
for ax in (top, bottom):
    ax.axvspan(DRIFT_START_WEEK - 0.5, N_WEEKS + 0.5, color="orange", alpha=0.08)
plt.show()

# A new hospital

The first scenario changed only the case mix. Now the patients come from
another hospital - VA Long Beach, one of the four of the data set - with
other habits and other gaps in the records. The deployed pipeline fills the
gaps with the Cleveland medians, so the model never sees them; you can.

In [ ]:
# The same data set was collected in four hospitals. The other three files are
# in the course repository as well - downloaded once.
for url in ["https://raw.githubusercontent.com/tomasvicar/AUI-public/master/labs/robustness/data/heart_disease_hungarian.csv",
            "https://raw.githubusercontent.com/tomasvicar/AUI-public/master/labs/robustness/data/heart_disease_switzerland.csv",
            "https://raw.githubusercontent.com/tomasvicar/AUI-public/master/labs/robustness/data/heart_disease_va_long_beach.csv"]:
    name = url.rsplit("/", 1)[1]
    if not Path(name).is_file():
        urllib.request.urlretrieve(url, name)
DATA = Path(".")

SITES = {"Cleveland": "heart_disease_cleveland.csv",
         "Hungary": "heart_disease_hungarian.csv",
         "Switzerland": "heart_disease_switzerland.csv",
         "VA Long Beach": "heart_disease_va_long_beach.csv"}
OTHER_SITES = ["Hungary", "Switzerland", "VA Long Beach"]


def load_site_raw(site: str) -> tuple[pd.DataFrame, pd.Series]:
    """One hospital as recorded: missing values left as NaN. A cholesterol or a
    resting blood pressure of 0 means "not measured" (all of Switzerland, 49
    VA patients) and becomes NaN as well."""
    table = pd.read_csv(DATA / SITES[site])
    y = (table.pop("num") > 0).astype(int)
    for column in ("chol", "trestbps"):
        table.loc[table[column] == 0, column] = np.nan
    return table, y


def cleveland_medians() -> pd.Series:
    return load_site_raw("Cleveland")[0].median()


def auc(p: np.ndarray, y: np.ndarray) -> float:
    """Area under the ROC curve: the probability that a random sick patient
    scores above a random healthy one (ties count one half)."""
    sick, healthy = p[y == 1], p[y == 0]
    greater = (sick[:, None] > healthy[None, :]).mean()
    ties = (sick[:, None] == healthy[None, :]).mean()
    return float(greater + 0.5 * ties)

In [ ]:
# The second scenario. From week 6 a new hospital - VA Long Beach - sends a
# growing share of the 60 weekly patients: 20 % in week 6, 100 % in week 10.
# `raw_rows` keeps every patient as recorded, before the Cleveland medians fill
# the gaps.
NEW_SITE = "VA Long Beach"
NEW_SITE_STEP = 0.20


def new_site_share(week: int) -> float:
    return min(1.0, NEW_SITE_STEP * max(0, week - DRIFT_START_WEEK + 1))


raw_new, y_new = load_site_raw(NEW_SITE)
X_new, y_new = raw_new.fillna(cleveland_medians()), y_new.to_numpy()
rng = np.random.RandomState(SEED)
batches_new = []
for week in range(1, N_WEEKS + 1):
    n_new = int(round(BATCH_SIZE * new_site_share(week)))
    old = rng.choice(len(X_test), BATCH_SIZE - n_new, replace=True)
    new = rng.choice(len(X_new), n_new, replace=True)
    batch = pd.concat([X_test.iloc[old], X_new.iloc[new]])
    labels = np.r_[y_pool[old], y_new[new]]
    raw_rows = pd.concat([X_test.iloc[old], raw_new.iloc[new]])
    batches_new.append((batch, labels, raw_rows))
print(f"{len(batches_new)} weeks; VA patients per week:",
      [int(round(BATCH_SIZE * new_site_share(w))) for w in range(1, N_WEEKS + 1)])

### Hole 3 - three monitors

In [ ]:
# TODO 3: the same dashboard for the new hospital, plus one more monitor: the
#         share of patients who arrive with at least one empty field
#         (`raw_rows.isna().any(axis=1)`). One dict per week in `weeks_new` with
#         the keys {"week", "psi", "status", "ks_p", "accuracy", "with_gaps"}.

weeks_new = []
for week, (batch, labels, raw_rows) in enumerate(batches_new, start=1):
    ...

In [ ]:
w = [x["week"] for x in weeks_new]
fig, (top, middle, bottom) = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
top.bar(w, [x["psi"] for x in weeks_new], color=[colors[x["status"]] for x in weeks_new])
top.axhline(WATCH, color="k", ls="--", lw=1); top.axhline(ALERT, color="k", ls="--", lw=1)
top.set_ylabel("PSI")
middle.semilogy(w, [x["ks_p"] for x in weeks_new], "o-k")
middle.axhline(0.05, color="red", ls="--", lw=1); middle.set_ylabel("K-S p-value")
bottom.bar(w, [x["with_gaps"] for x in weeks_new], color="navy")
bottom.set_ylabel("share with an empty field"); bottom.set_xticks(w)
for ax in (top, middle, bottom):
    ax.axvspan(DRIFT_START_WEEK - 0.5, N_WEEKS + 0.5, color="orange", alpha=0.08)
plt.show()

# Questions

1. In which week would you have called somebody? On PSI, on the p-value, or on
   both?
2. Week 9 has a lower PSI than week 8 although the drift is stronger. Why?
3. The accuracy hardly moves while PSI alarms. Is the alarm useless?
4. PSI 0.20 and p = 0.02: statistically significant, practically "watch". With
   batches of 600 patients instead of 60, which of the two numbers would
   change more - and why?
5. The new hospital: which of the three monitors rings first, and in which
   week? Why does the cheapest one win?
6. This time the accuracy does fall. What is different from the first
   scenario?

In [ ]:
# Space for your answers.

---

*The numbers are computed by [`heart.py`](https://github.com/tomasvicar/AUI-public/blob/master/lectures/robustness/code/heart.py) of
the lecture. The notebook is built by
[`code/build_notebooks.py`](https://github.com/tomasvicar/AUI/blob/master/labs/robustness/code/build_notebooks.py) - editing the
`.ipynb` by hand gets overwritten.*